In [1]:
import os, random, time, json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from imblearn.metrics import specificity_score
from mambapy.vim import VMamba, MambaConfig
from thop import profile

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:
AUG_ROOT = "D:/mamba_model/aug_clean_tio"
TAG      = "tio"
# ───────────────────────────────────────────────────────────────

COHORT_CSV = "D:/mamba_model/thesis_cohort_clean.csv"
MRI_CACHE  = f"{AUG_ROOT}/roi_mri"
PET_CACHE  = f"{AUG_ROOT}/roi_pet"
CKPT_DIR   = f"D:/mamba_model/checkpoints_v7_region_{TAG}"
RESULTS    = f"D:/mamba_model/v7_region_{TAG}_results.json"
os.makedirs(CKPT_DIR, exist_ok=True)

SPLIT_SEED  = 42
AUG_SEEDS   = [1, 101, 42]
BATCH_SIZE  = 4
NUM_WORKERS = 0

REGION_PAIRS = {
    "hippocampus":   [0, 1],
    "cerebellum_wm": [2, 3],
    "cerebral_wm":   [4, 5],
}
ALL_NAMES = ['L-Hippo', 'R-Hippo', 'L-Cereb-WM',
             'R-Cereb-WM', 'L-Cerebral-WM', 'R-Cerebral-WM']

print(f"aug:  {AUG_ROOT}")
print(f"ckpt: {CKPT_DIR}")
for p in (MRI_CACHE, PET_CACHE):
    n = len(os.listdir(p)) if os.path.isdir(p) else 0
    print(f"  {os.path.basename(p)}: {n} files{'  *** MISSING ***' if n == 0 else ''}")
print()
for name, idx in REGION_PAIRS.items():
    print(f"  {name:15s} indices {idx} -> {[ALL_NAMES[i] for i in idx]}")

aug:  D:/mamba_model/aug_clean_tio
ckpt: D:/mamba_model/checkpoints_v7_region_tio
  roi_mri: 560 files
  roi_pet: 560 files

  hippocampus     indices [0, 1] -> ['L-Hippo', 'R-Hippo']
  cerebellum_wm   indices [2, 3] -> ['L-Cereb-WM', 'R-Cereb-WM']
  cerebral_wm     indices [4, 5] -> ['L-Cerebral-WM', 'R-Cerebral-WM']


In [3]:
class VimEncoder(nn.Module):
    """Bidirectional Mamba over a token sequence. No CNN, no pretraining."""
    def __init__(self, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        cfg = MambaConfig(d_model=d_model, n_layers=n_layers, d_state=d_state,
                          bidirectional=True, divide_output=True,
                          pscan=True, use_cuda=False)
        self.encoder = VMamba(cfg)
        self.final_norm = nn.LayerNorm(d_model)
    def forward(self, tokens):
        return self.final_norm(self.encoder(tokens))

In [4]:
class ROIPatchEmbed3D(nn.Module):
    """6 ROIs -> non-overlapping 8^3 patches -> one token each. 3072 tokens."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32):
        super().__init__()
        self.n_rois, self.patch_size = n_rois, patch_size
        self.grid_size = roi_size // patch_size
        self.patches_per_roi = self.grid_size ** 3
        self.d_model = d_model
        self.patch_conv = nn.Conv3d(1, d_model, kernel_size=patch_size, stride=patch_size)
        self.roi_embed    = nn.Embedding(n_rois, d_model)
        self.depth_embed  = nn.Embedding(self.grid_size, d_model)
        self.height_embed = nn.Embedding(self.grid_size, d_model)
        self.width_embed  = nn.Embedding(self.grid_size, d_model)
        with torch.no_grad():
            for e in [self.roi_embed, self.depth_embed, self.height_embed, self.width_embed]:
                e.weight.mul_(0.02)
        d, h, w = torch.meshgrid(torch.arange(self.grid_size), torch.arange(self.grid_size),
                                 torch.arange(self.grid_size), indexing="ij")
        self.register_buffer("coordinates", torch.stack([d, h, w], -1).reshape(-1, 3),
                             persistent=False)

    def forward(self, rois):
        B, n = rois.shape[:2]
        x = rois.reshape(B * n, 1, *rois.shape[-3:])
        tokens = self.patch_conv(x).flatten(2).transpose(1, 2)
        tokens = tokens.reshape(B, n, self.patches_per_roi, self.d_model)
        c = self.coordinates
        spatial = (self.depth_embed(c[:, 0]) + self.height_embed(c[:, 1])
                   + self.width_embed(c[:, 2]))
        tokens = tokens + spatial[None, None] + self.roi_embed.weight[None, :, None, :]
        occ = F.max_pool3d((x.abs() > 1e-6).float(),
                           kernel_size=self.patch_size, stride=self.patch_size)
        valid = occ.flatten(1).bool().reshape(B, n, self.patches_per_roi)
        tokens = tokens.reshape(B, -1, self.d_model)
        valid = valid.reshape(B, -1)
        return tokens * valid.unsqueeze(-1).to(tokens.dtype), valid


class VisionMambaBranch(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32,
                 n_layers=2, d_state=16):
        super().__init__()
        self.n_rois = n_rois
        self.patch_embed = ROIPatchEmbed3D(n_rois, roi_size, patch_size, d_model)
        self.vim = VimEncoder(d_model, n_layers, d_state)
    def forward(self, rois):
        tokens, valid = self.patch_embed(rois)
        tokens = self.vim(tokens)
        w = valid.unsqueeze(-1).to(tokens.dtype)
        return (tokens * w).sum(1) / w.sum(1).clamp_min(1.0)


class VisionMambaModel(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32,
                 n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, n_classes)
    def forward(self, rois):
        return self.classifier(self.dropout(self.branch(rois)))


class MultimodalVisionMambaModel(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32,
                 n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.mri_branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.pet_branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model * 2, n_classes)
    def forward(self, mri, pet):
        f = torch.cat([self.mri_branch(mri), self.pet_branch(pet)], dim=1)
        return self.classifier(self.dropout(f))

In [5]:
df = pd.read_csv(COHORT_CSV)
sessions, labels = df["mri_session"].values, df["outcome_label"].values

X_tv, X_test, y_tv, y_test = train_test_split(
    sessions, labels, test_size=0.2, random_state=SPLIT_SEED, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=SPLIT_SEED, stratify=y_tv)

session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))
print(f"train {len(X_train)} | val {len(X_val)} | test {len(X_test)} "
      f"| test pos {int(y_test.sum())}")


# in-memory cache, shared across all three region pairs 
# Stores the full 6-ROI array once per file; each dataset slices its own pair.
_CACHE = {}

def load_cached(path):
    a = _CACHE.get(path)
    if a is None:
        a = np.load(path).astype(np.float32)
        _CACHE[path] = a
    return a



class RegionDataset(Dataset):
    def __init__(self, sessions, labels, cache_dir, region_idx,
                 is_mri=True, is_train=False):
        self.samples, self.cache_dir, self.idx = [], cache_dir, region_idx
        for ses, lab in zip(sessions, labels):
            key = ses if is_mri else session_to_subject[ses]
            self.samples.append((key, lab, "orig"))
            if is_train:
                for s in AUG_SEEDS:
                    self.samples.append((key, lab, f"aug{s}"))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        key, lab, ver = self.samples[i]
        a = load_cached(f"{self.cache_dir}/{key}_{ver}.npy")[self.idx].copy()
        return torch.from_numpy(a).unsqueeze(1), torch.tensor(lab, dtype=torch.long), key


class MultimodalRegionDataset(Dataset):
    def __init__(self, sessions, labels, mri_dir, pet_dir, region_idx, is_train=False):
        self.samples, self.mri_dir, self.pet_dir, self.idx = [], mri_dir, pet_dir, region_idx
        for ses, lab in zip(sessions, labels):
            sid = session_to_subject[ses]
            self.samples.append((ses, sid, lab, "orig"))
            if is_train:
                for s in AUG_SEEDS:
                    self.samples.append((ses, sid, lab, f"aug{s}"))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        mk, pk, lab, ver = self.samples[i]
        m = load_cached(f"{self.mri_dir}/{mk}_{ver}.npy")[self.idx].copy()
        p = load_cached(f"{self.pet_dir}/{pk}_{ver}.npy")[self.idx].copy()
        return (torch.from_numpy(m).unsqueeze(1), torch.from_numpy(p).unsqueeze(1),
                torch.tensor(lab, dtype=torch.long), mk)


def dl(ds, shuffle=False):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      persistent_workers=(NUM_WORKERS > 0))


def make_loaders(region_idx):
    mri = (dl(RegionDataset(X_train, y_train, MRI_CACHE, region_idx, True, True), True),
           dl(RegionDataset(X_val,   y_val,   MRI_CACHE, region_idx, True, False)),
           dl(RegionDataset(X_test,  y_test,  MRI_CACHE, region_idx, True, False)))
    pet = (dl(RegionDataset(X_train, y_train, PET_CACHE, region_idx, False, True), True),
           dl(RegionDataset(X_val,   y_val,   PET_CACHE, region_idx, False, False)),
           dl(RegionDataset(X_test,  y_test,  PET_CACHE, region_idx, False, False)))
    mm  = (dl(MultimodalRegionDataset(X_train, y_train, MRI_CACHE, PET_CACHE, region_idx, True), True),
           dl(MultimodalRegionDataset(X_val,   y_val,   MRI_CACHE, PET_CACHE, region_idx, False)),
           dl(MultimodalRegionDataset(X_test,  y_test,  MRI_CACHE, PET_CACHE, region_idx, False)))
    return mri, pet, mm


LOADERS = {name: make_loaders(idx) for name, idx in REGION_PAIRS.items()}

b = next(iter(LOADERS['hippocampus'][0][0]))
print(f"\nbatch shape: {tuple(b[0].shape)}  (expect ({BATCH_SIZE}, 2, 1, 64, 64, 64))")

t0 = time.time()
for i, _ in enumerate(LOADERS['hippocampus'][0][0]):
    if i >= 20: break
cold = time.time() - t0
t0 = time.time()
for i, _ in enumerate(LOADERS['hippocampus'][0][0]):
    if i >= 20: break
warm = time.time() - t0
print(f"20 batches: cold {cold:.1f}s -> warm {warm:.1f}s")
print(f"cached arrays: {len(_CACHE)} (~{sum(a.nbytes for a in _CACHE.values())/1e9:.1f} GB)")

train 120 | val 40 | test 40 | test pos 20

batch shape: (4, 2, 1, 64, 64, 64)  (expect (4, 2, 1, 64, 64, 64))
20 batches: cold 7.0s -> warm 5.8s
cached arrays: 156 (~1.0 GB)


In [6]:
def train_epoch(model, loader, opt, crit, mm):
    model.train(); tot = 0
    for batch in loader:
        opt.zero_grad()
        if mm:
            a, b, lb, _ = batch; out = model(a.to(device), b.to(device))
        else:
            a, lb, _ = batch;    out = model(a.to(device))
        loss = crit(out, lb.to(device)); loss.backward(); opt.step(); tot += loss.item()
    return tot / len(loader)

def evaluate(model, loader, crit, mm):
    model.eval(); tot, P, L = 0, [], []
    with torch.no_grad():
        for batch in loader:
            if mm:
                a, b, lb, _ = batch; out = model(a.to(device), b.to(device))
            else:
                a, lb, _ = batch;    out = model(a.to(device))
            tot += crit(out, lb.to(device)).item()
            P.extend(out.argmax(1).cpu().numpy()); L.extend(lb.numpy())
    return (tot / len(loader), np.mean(np.array(P) == np.array(L)),
            recall_score(L, P, zero_division=0), specificity_score(L, P))

def measure_inference(model, loader, mm, n=20):
    model.eval(); ts = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n: break
            if mm:
                a, b = batch[0].to(device), batch[1].to(device); bs = a.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time(); _ = model(a, b)
            else:
                a = batch[0].to(device); bs = a.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time(); _ = model(a)
            if device.type == 'cuda': torch.cuda.synchronize()
            ts.append((time.time() - t0) / bs)
    return np.mean(ts), np.std(ts)

def compute_flops(model, loader, mm):
    try:
        model.eval(); b = next(iter(loader))
        with torch.no_grad():
            inp = (b[0][:1].to(device), b[1][:1].to(device)) if mm else (b[0][:1].to(device),)
            macs, _ = profile(model, inputs=inp, verbose=False)
        return macs * 2
    except Exception as e:
        print(f"  (FLOPs failed: {e})"); return None


def run_seed(seed, model_cls, loaders, mm, prefix,
             max_epochs=101, patience=15, min_epochs=25, lr=1e-4, log_every=10):
    torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    np.random.seed(seed); random.seed(seed)
    tr, va, te = loaders

    # n_rois=2 -- one bilateral pair
    model = model_cls(n_rois=2, d_model=32, n_layers=2,
                      n_classes=2, dropout=0.4).to(device)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    opt  = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    sch  = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=10)

    best, no_imp, best_ep, total = float('inf'), 0, 0, 0
    path = f"{CKPT_DIR}/{prefix}_seed{seed}.pt"
    print(f"\n--- {prefix} seed {seed} ---")

    for ep in range(1, max_epochs):
        t0 = time.time()
        trl = train_epoch(model, tr, opt, crit, mm)
        vl, vacc, vtpr, vtnr = evaluate(model, va, crit, mm)
        sch.step(vl); dt = time.time() - t0; total += dt
        if ep % log_every == 0 or ep == 1:
            print(f"  ep {ep:>3} | train {trl:.4f} | val {vl:.4f} | "
                  f"acc {vacc:.3f} tpr {vtpr:.3f} tnr {vtnr:.3f} | {dt:.0f}s")
        if vl < best:
            best, best_ep, no_imp = vl, ep, 0
            torch.save(model.state_dict(), path)
        else:
            no_imp += 1
            if ep >= min_epochs and no_imp >= patience:
                print(f"  early stop {ep}, best {best_ep}"); break

    if best_ep < 5:
        print(f"  WARNING: best epoch {best_ep} -- may never have left initialisation")

    model.load_state_dict(torch.load(path, weights_only=True))
    _, acc, tpr, tnr = evaluate(model, te, crit, mm)
    npar = sum(p.numel() for p in model.parameters() if p.requires_grad)
    inf_m, inf_s = measure_inference(model, te, mm)
    fl = compute_flops(model, te, mm)

    print(f"  >>> TEST Acc={acc*100:.1f}% TPR={tpr*100:.1f}% TNR={tnr*100:.1f}% | "
          f"params={npar:,} train={total/60:.1f}min inf={inf_m*1000:.2f}ms "
          f"{f'{fl/1e9:.3f}GFLOPs' if fl else ''} best_ep={best_ep}")

    return {"seed": seed, "acc": acc, "tpr": tpr, "tnr": tnr, "best_epoch": best_ep,
            "train_time_sec": total, "n_params": npar,
            "inf_time_ms": inf_m * 1000, "inf_std_ms": inf_s * 1000, "flops": fl}


results = {f"{r}_{m}": [] for r in REGION_PAIRS for m in ("mri", "pet", "mm")}
print("ready —", len(results), "conditions x 4 seeds = 36 runs")

ready — 9 conditions x 4 seeds = 36 runs


In [7]:
results["hippocampus_mri"].append(
    run_seed(1, VisionMambaModel, LOADERS["hippocampus"][0], False, "v7_hippo_mri"))


--- v7_hippo_mri seed 1 ---
  ep   1 | train 0.6952 | val 0.6859 | acc 0.600 tpr 0.550 tnr 0.650 | 37s
  ep  10 | train 0.6642 | val 0.6617 | acc 0.625 tpr 0.600 tnr 0.650 | 3s
  ep  20 | train 0.5733 | val 0.6191 | acc 0.575 tpr 0.600 tnr 0.550 | 3s
  ep  30 | train 0.3653 | val 0.6577 | acc 0.600 tpr 0.650 tnr 0.550 | 3s
  early stop 37, best 22
  >>> TEST Acc=62.5% TPR=60.0% TNR=65.0% | params=44,834 train=2.6min inf=3.95ms 0.079GFLOPs best_ep=22


In [8]:
results["hippocampus_mri"].append(
    run_seed(7, VisionMambaModel, LOADERS["hippocampus"][0], False, "v7_hippo_mri"))


--- v7_hippo_mri seed 7 ---
  ep   1 | train 0.7070 | val 0.6908 | acc 0.500 tpr 1.000 tnr 0.000 | 3s
  ep  10 | train 0.6653 | val 0.6683 | acc 0.550 tpr 0.950 tnr 0.150 | 3s
  ep  20 | train 0.5867 | val 0.6164 | acc 0.600 tpr 0.600 tnr 0.600 | 3s
  ep  30 | train 0.4194 | val 0.6017 | acc 0.700 tpr 0.600 tnr 0.800 | 3s
  ep  40 | train 0.2494 | val 0.5919 | acc 0.700 tpr 0.650 tnr 0.750 | 3s
  early stop 46, best 31
  >>> TEST Acc=62.5% TPR=65.0% TNR=60.0% | params=44,834 train=2.5min inf=2.20ms 0.079GFLOPs best_ep=31


In [9]:
results["hippocampus_mri"].append(
    run_seed(123, VisionMambaModel, LOADERS["hippocampus"][0], False, "v7_hippo_mri"))


--- v7_hippo_mri seed 123 ---
  ep   1 | train 0.6931 | val 0.6822 | acc 0.600 tpr 0.650 tnr 0.550 | 3s
  ep  10 | train 0.6633 | val 0.6585 | acc 0.700 tpr 0.550 tnr 0.850 | 3s
  ep  20 | train 0.5753 | val 0.6203 | acc 0.775 tpr 0.550 tnr 1.000 | 3s
  ep  30 | train 0.3824 | val 0.6376 | acc 0.775 tpr 0.600 tnr 0.950 | 3s
  early stop 36, best 21
  >>> TEST Acc=60.0% TPR=75.0% TNR=45.0% | params=44,834 train=2.0min inf=2.05ms 0.079GFLOPs best_ep=21


In [11]:
results["hippocampus_pet"].append(
    run_seed(1, VisionMambaModel, LOADERS["hippocampus"][1], False, "v7_hippo_pet"))


--- v7_hippo_pet seed 1 ---
  ep   1 | train 0.6958 | val 0.6867 | acc 0.650 tpr 0.550 tnr 0.750 | 47s
  ep  10 | train 0.6596 | val 0.6573 | acc 0.700 tpr 0.600 tnr 0.800 | 3s
  ep  20 | train 0.5296 | val 0.5689 | acc 0.750 tpr 0.750 tnr 0.750 | 3s
  ep  30 | train 0.3542 | val 0.5800 | acc 0.700 tpr 0.650 tnr 0.750 | 3s
  early stop 37, best 22
  >>> TEST Acc=65.0% TPR=70.0% TNR=60.0% | params=44,834 train=2.7min inf=3.95ms 0.079GFLOPs best_ep=22


In [12]:
results["hippocampus_pet"].append(
    run_seed(7, VisionMambaModel, LOADERS["hippocampus"][1], False, "v7_hippo_pet"))


--- v7_hippo_pet seed 7 ---
  ep   1 | train 0.7060 | val 0.6894 | acc 0.500 tpr 1.000 tnr 0.000 | 3s
  ep  10 | train 0.6641 | val 0.6609 | acc 0.600 tpr 0.950 tnr 0.250 | 3s
  ep  20 | train 0.5605 | val 0.5881 | acc 0.750 tpr 0.800 tnr 0.700 | 3s
  ep  30 | train 0.3929 | val 0.6031 | acc 0.650 tpr 0.450 tnr 0.850 | 3s
  ep  40 | train 0.2521 | val 0.6989 | acc 0.650 tpr 0.400 tnr 0.900 | 3s
  early stop 40, best 25
  >>> TEST Acc=65.0% TPR=45.0% TNR=85.0% | params=44,834 train=2.2min inf=1.94ms 0.079GFLOPs best_ep=25


In [13]:
results["hippocampus_pet"].append(
    run_seed(123, VisionMambaModel, LOADERS["hippocampus"][1], False, "v7_hippo_pet"))


--- v7_hippo_pet seed 123 ---
  ep   1 | train 0.6936 | val 0.6883 | acc 0.625 tpr 0.500 tnr 0.750 | 3s
  ep  10 | train 0.6550 | val 0.6552 | acc 0.675 tpr 0.500 tnr 0.850 | 3s
  ep  20 | train 0.5335 | val 0.6259 | acc 0.675 tpr 0.450 tnr 0.900 | 3s
  ep  30 | train 0.3719 | val 0.6230 | acc 0.725 tpr 0.600 tnr 0.850 | 3s
  ep  40 | train 0.2292 | val 0.6737 | acc 0.700 tpr 0.550 tnr 0.850 | 3s
  early stop 41, best 26
  >>> TEST Acc=70.0% TPR=70.0% TNR=70.0% | params=44,834 train=2.2min inf=1.96ms 0.079GFLOPs best_ep=26


In [15]:
results["hippocampus_mm"].append(
    run_seed(1, MultimodalVisionMambaModel, LOADERS["hippocampus"][2], True, "v7_hippo_mm"))


--- v7_hippo_mm seed 1 ---
  ep   1 | train 0.7017 | val 0.6810 | acc 0.500 tpr 1.000 tnr 0.000 | 6s
  ep  10 | train 0.6490 | val 0.6383 | acc 0.675 tpr 0.700 tnr 0.650 | 6s
  ep  20 | train 0.4860 | val 0.5717 | acc 0.700 tpr 0.550 tnr 0.850 | 6s
  ep  30 | train 0.2373 | val 0.5741 | acc 0.725 tpr 0.550 tnr 0.900 | 6s
  ep  40 | train 0.1407 | val 0.5823 | acc 0.725 tpr 0.550 tnr 0.900 | 6s
  early stop 44, best 29
  >>> TEST Acc=72.5% TPR=65.0% TNR=80.0% | params=89,666 train=4.7min inf=3.71ms 0.158GFLOPs best_ep=29


In [16]:
results["hippocampus_mm"].append(
    run_seed(7, MultimodalVisionMambaModel, LOADERS["hippocampus"][2], True, "v7_hippo_mm"))


--- v7_hippo_mm seed 7 ---
  ep   1 | train 0.7058 | val 0.6863 | acc 0.550 tpr 1.000 tnr 0.100 | 6s
  ep  10 | train 0.6471 | val 0.6428 | acc 0.675 tpr 0.700 tnr 0.650 | 6s
  ep  20 | train 0.5045 | val 0.5949 | acc 0.725 tpr 0.500 tnr 0.950 | 6s
  ep  30 | train 0.2958 | val 0.5767 | acc 0.700 tpr 0.550 tnr 0.850 | 6s
  early stop 39, best 24
  >>> TEST Acc=72.5% TPR=80.0% TNR=65.0% | params=89,666 train=4.2min inf=3.74ms 0.158GFLOPs best_ep=24


In [17]:
results["hippocampus_mm"].append(
    run_seed(123, MultimodalVisionMambaModel, LOADERS["hippocampus"][2], True, "v7_hippo_mm"))


--- v7_hippo_mm seed 123 ---
  ep   1 | train 0.7036 | val 0.6895 | acc 0.500 tpr 1.000 tnr 0.000 | 6s
  ep  10 | train 0.6531 | val 0.6532 | acc 0.625 tpr 0.800 tnr 0.450 | 6s
  ep  20 | train 0.5060 | val 0.5895 | acc 0.750 tpr 0.550 tnr 0.950 | 6s
  ep  30 | train 0.2940 | val 0.5686 | acc 0.700 tpr 0.600 tnr 0.800 | 6s
  ep  40 | train 0.1651 | val 0.5931 | acc 0.750 tpr 0.600 tnr 0.900 | 6s
  early stop 40, best 25
  >>> TEST Acc=72.5% TPR=70.0% TNR=75.0% | params=89,666 train=4.3min inf=3.76ms 0.158GFLOPs best_ep=25


In [19]:
results["cerebellum_wm_mri"].append(
    run_seed(1, VisionMambaModel, LOADERS["cerebellum_wm"][0], False, "v7_cerebwm_mri"))


--- v7_cerebwm_mri seed 1 ---
  ep   1 | train 0.6956 | val 0.6870 | acc 0.525 tpr 0.300 tnr 0.750 | 4s
  ep  10 | train 0.6640 | val 0.6602 | acc 0.650 tpr 0.500 tnr 0.800 | 3s
  ep  20 | train 0.5677 | val 0.6127 | acc 0.550 tpr 0.550 tnr 0.550 | 3s
  ep  30 | train 0.4021 | val 0.5993 | acc 0.600 tpr 0.550 tnr 0.650 | 3s
  ep  40 | train 0.1992 | val 0.6159 | acc 0.600 tpr 0.550 tnr 0.650 | 3s
  ep  50 | train 0.1336 | val 0.6188 | acc 0.650 tpr 0.450 tnr 0.850 | 3s
  ep  60 | train 0.1267 | val 0.6110 | acc 0.725 tpr 0.550 tnr 0.900 | 3s
  ep  70 | train 0.1233 | val 0.6347 | acc 0.675 tpr 0.450 tnr 0.900 | 3s
  early stop 72, best 57
  >>> TEST Acc=52.5% TPR=55.0% TNR=50.0% | params=44,834 train=4.0min inf=2.09ms 0.079GFLOPs best_ep=57


In [20]:
results["cerebellum_wm_mri"].append(
    run_seed(7, VisionMambaModel, LOADERS["cerebellum_wm"][0], False, "v7_cerebwm_mri"))


--- v7_cerebwm_mri seed 7 ---
  ep   1 | train 0.7028 | val 0.6883 | acc 0.525 tpr 1.000 tnr 0.050 | 3s
  ep  10 | train 0.6585 | val 0.6571 | acc 0.575 tpr 0.950 tnr 0.200 | 3s
  ep  20 | train 0.5717 | val 0.6055 | acc 0.675 tpr 0.800 tnr 0.550 | 3s
  ep  30 | train 0.3948 | val 0.5724 | acc 0.625 tpr 0.600 tnr 0.650 | 3s
  ep  40 | train 0.2130 | val 0.5876 | acc 0.625 tpr 0.600 tnr 0.650 | 3s
  ep  50 | train 0.1383 | val 0.6348 | acc 0.675 tpr 0.600 tnr 0.750 | 3s
  early stop 50, best 35
  >>> TEST Acc=57.5% TPR=50.0% TNR=65.0% | params=44,834 train=2.8min inf=2.04ms 0.079GFLOPs best_ep=35


In [21]:
results["cerebellum_wm_mri"].append(
    run_seed(123, VisionMambaModel, LOADERS["cerebellum_wm"][0], False, "v7_cerebwm_mri"))


--- v7_cerebwm_mri seed 123 ---
  ep   1 | train 0.6971 | val 0.6846 | acc 0.675 tpr 0.750 tnr 0.600 | 3s
  ep  10 | train 0.6574 | val 0.6554 | acc 0.600 tpr 0.300 tnr 0.900 | 3s
  ep  20 | train 0.5675 | val 0.6337 | acc 0.600 tpr 0.350 tnr 0.850 | 3s
  ep  30 | train 0.3933 | val 0.6796 | acc 0.575 tpr 0.300 tnr 0.850 | 3s
  ep  40 | train 0.2018 | val 0.6578 | acc 0.650 tpr 0.600 tnr 0.700 | 3s
  early stop 42, best 27
  >>> TEST Acc=67.5% TPR=65.0% TNR=70.0% | params=44,834 train=2.3min inf=2.12ms 0.079GFLOPs best_ep=27


In [23]:
results["cerebellum_wm_pet"].append(
    run_seed(1, VisionMambaModel, LOADERS["cerebellum_wm"][1], False, "v7_cerebwm_pet"))


--- v7_cerebwm_pet seed 1 ---
  ep   1 | train 0.6992 | val 0.6915 | acc 0.525 tpr 0.050 tnr 1.000 | 3s
  ep  10 | train 0.6714 | val 0.6795 | acc 0.575 tpr 0.300 tnr 0.850 | 3s
  ep  20 | train 0.5596 | val 0.5999 | acc 0.725 tpr 0.650 tnr 0.800 | 3s
  ep  30 | train 0.3758 | val 0.5999 | acc 0.725 tpr 0.700 tnr 0.750 | 3s
  ep  40 | train 0.2124 | val 0.6778 | acc 0.725 tpr 0.700 tnr 0.750 | 3s
  early stop 44, best 29
  >>> TEST Acc=60.0% TPR=55.0% TNR=65.0% | params=44,834 train=2.4min inf=1.95ms 0.079GFLOPs best_ep=29


In [24]:
results["cerebellum_wm_pet"].append(
    run_seed(7, VisionMambaModel, LOADERS["cerebellum_wm"][1], False, "v7_cerebwm_pet"))


--- v7_cerebwm_pet seed 7 ---
  ep   1 | train 0.7042 | val 0.6929 | acc 0.500 tpr 1.000 tnr 0.000 | 3s
  ep  10 | train 0.6723 | val 0.6785 | acc 0.600 tpr 1.000 tnr 0.200 | 3s
  ep  20 | train 0.5867 | val 0.6410 | acc 0.650 tpr 0.900 tnr 0.400 | 3s
  ep  30 | train 0.4051 | val 0.5961 | acc 0.700 tpr 0.550 tnr 0.850 | 3s
  ep  40 | train 0.2387 | val 0.6521 | acc 0.650 tpr 0.650 tnr 0.650 | 3s
  early stop 45, best 30
  >>> TEST Acc=60.0% TPR=45.0% TNR=75.0% | params=44,834 train=2.5min inf=1.92ms 0.079GFLOPs best_ep=30


In [25]:
results["cerebellum_wm_pet"].append(
    run_seed(123, VisionMambaModel, LOADERS["cerebellum_wm"][1], False, "v7_cerebwm_pet"))


--- v7_cerebwm_pet seed 123 ---
  ep   1 | train 0.7006 | val 0.6911 | acc 0.625 tpr 0.600 tnr 0.650 | 3s
  ep  10 | train 0.6663 | val 0.6682 | acc 0.600 tpr 0.350 tnr 0.850 | 3s
  ep  20 | train 0.5589 | val 0.6313 | acc 0.625 tpr 0.350 tnr 0.900 | 3s
  ep  30 | train 0.3958 | val 0.6294 | acc 0.675 tpr 0.500 tnr 0.850 | 3s
  ep  40 | train 0.2282 | val 0.6471 | acc 0.700 tpr 0.650 tnr 0.750 | 3s
  early stop 41, best 26
  >>> TEST Acc=65.0% TPR=55.0% TNR=75.0% | params=44,834 train=2.2min inf=1.94ms 0.079GFLOPs best_ep=26


In [27]:
results["cerebellum_wm_mm"].append(
    run_seed(1, MultimodalVisionMambaModel, LOADERS["cerebellum_wm"][2], True, "v7_cerebwm_mm"))


--- v7_cerebwm_mm seed 1 ---
  ep   1 | train 0.7022 | val 0.6914 | acc 0.500 tpr 1.000 tnr 0.000 | 6s
  ep  10 | train 0.6461 | val 0.6446 | acc 0.625 tpr 0.650 tnr 0.600 | 6s
  ep  20 | train 0.4878 | val 0.5979 | acc 0.725 tpr 0.650 tnr 0.800 | 6s
  ep  30 | train 0.2432 | val 0.6523 | acc 0.675 tpr 0.700 tnr 0.650 | 6s
  early stop 34, best 19
  >>> TEST Acc=57.5% TPR=50.0% TNR=65.0% | params=89,666 train=3.6min inf=3.71ms 0.158GFLOPs best_ep=19


In [28]:
results["cerebellum_wm_mm"].append(
    run_seed(7, MultimodalVisionMambaModel, LOADERS["cerebellum_wm"][2], True, "v7_cerebwm_mm"))


--- v7_cerebwm_mm seed 7 ---
  ep   1 | train 0.7015 | val 0.6917 | acc 0.500 tpr 1.000 tnr 0.000 | 6s
  ep  10 | train 0.6474 | val 0.6488 | acc 0.650 tpr 0.700 tnr 0.600 | 6s
  ep  20 | train 0.5038 | val 0.5953 | acc 0.725 tpr 0.550 tnr 0.900 | 6s
  ep  30 | train 0.2801 | val 0.5290 | acc 0.725 tpr 0.650 tnr 0.800 | 6s
  ep  40 | train 0.1388 | val 0.5540 | acc 0.725 tpr 0.700 tnr 0.750 | 6s
  early stop 45, best 30
  >>> TEST Acc=65.0% TPR=65.0% TNR=65.0% | params=89,666 train=4.8min inf=3.76ms 0.158GFLOPs best_ep=30


In [29]:
results["cerebellum_wm_mm"].append(
    run_seed(123, MultimodalVisionMambaModel, LOADERS["cerebellum_wm"][2], True, "v7_cerebwm_mm"))


--- v7_cerebwm_mm seed 123 ---
  ep   1 | train 0.7027 | val 0.6896 | acc 0.500 tpr 1.000 tnr 0.000 | 6s
  ep  10 | train 0.6582 | val 0.6569 | acc 0.650 tpr 0.750 tnr 0.550 | 7s
  ep  20 | train 0.5110 | val 0.6141 | acc 0.725 tpr 0.600 tnr 0.850 | 6s
  ep  30 | train 0.2807 | val 0.6032 | acc 0.675 tpr 0.650 tnr 0.700 | 6s
  early stop 36, best 21
  >>> TEST Acc=60.0% TPR=65.0% TNR=55.0% | params=89,666 train=3.9min inf=3.85ms 0.158GFLOPs best_ep=21


In [31]:
results["cerebral_wm_mri"].append(
    run_seed(1, VisionMambaModel, LOADERS["cerebral_wm"][0], False, "v7_cerebralwm_mri"))


--- v7_cerebralwm_mri seed 1 ---
  ep   1 | train 0.6970 | val 0.6931 | acc 0.475 tpr 0.050 tnr 0.900 | 4s
  ep  10 | train 0.6775 | val 0.6811 | acc 0.650 tpr 0.350 tnr 0.950 | 3s
  ep  20 | train 0.6262 | val 0.6578 | acc 0.650 tpr 0.650 tnr 0.650 | 3s
  ep  30 | train 0.4518 | val 0.6690 | acc 0.550 tpr 0.800 tnr 0.300 | 3s
  ep  40 | train 0.2013 | val 0.7182 | acc 0.575 tpr 0.650 tnr 0.500 | 3s
  early stop 43, best 28
  >>> TEST Acc=62.5% TPR=60.0% TNR=65.0% | params=44,834 train=2.4min inf=1.95ms 0.079GFLOPs best_ep=28


In [32]:
results["cerebral_wm_mri"].append(
    run_seed(7, VisionMambaModel, LOADERS["cerebral_wm"][0], False, "v7_cerebralwm_mri"))


--- v7_cerebralwm_mri seed 7 ---
  ep   1 | train 0.7041 | val 0.6925 | acc 0.500 tpr 1.000 tnr 0.000 | 3s
  ep  10 | train 0.6841 | val 0.6833 | acc 0.500 tpr 1.000 tnr 0.000 | 3s
  ep  20 | train 0.6299 | val 0.6575 | acc 0.575 tpr 0.700 tnr 0.450 | 3s
  ep  30 | train 0.4609 | val 0.6724 | acc 0.625 tpr 0.650 tnr 0.600 | 3s
  early stop 39, best 24
  >>> TEST Acc=57.5% TPR=40.0% TNR=75.0% | params=44,834 train=2.1min inf=1.99ms 0.079GFLOPs best_ep=24


In [33]:
results["cerebral_wm_mri"].append(
    run_seed(123, VisionMambaModel, LOADERS["cerebral_wm"][0], False, "v7_cerebralwm_mri"))


--- v7_cerebralwm_mri seed 123 ---
  ep   1 | train 0.6973 | val 0.6962 | acc 0.425 tpr 0.400 tnr 0.450 | 3s
  ep  10 | train 0.6806 | val 0.6837 | acc 0.650 tpr 0.500 tnr 0.800 | 3s
  ep  20 | train 0.6278 | val 0.6599 | acc 0.700 tpr 0.500 tnr 0.900 | 3s
  ep  30 | train 0.4646 | val 0.6248 | acc 0.575 tpr 0.550 tnr 0.600 | 3s
  ep  40 | train 0.2128 | val 0.6690 | acc 0.575 tpr 0.550 tnr 0.600 | 3s
  early stop 46, best 31
  >>> TEST Acc=62.5% TPR=50.0% TNR=75.0% | params=44,834 train=2.5min inf=1.99ms 0.079GFLOPs best_ep=31


In [35]:
results["cerebral_wm_pet"].append(
    run_seed(1, VisionMambaModel, LOADERS["cerebral_wm"][1], False, "v7_cerebralwm_pet"))


--- v7_cerebralwm_pet seed 1 ---
  ep   1 | train 0.6902 | val 0.6687 | acc 0.675 tpr 0.400 tnr 0.950 | 3s
  ep  10 | train 0.6380 | val 0.6212 | acc 0.675 tpr 0.450 tnr 0.900 | 3s
  ep  20 | train 0.5417 | val 0.5660 | acc 0.775 tpr 0.700 tnr 0.850 | 3s
  ep  30 | train 0.3582 | val 0.5291 | acc 0.800 tpr 0.750 tnr 0.850 | 3s
  ep  40 | train 0.1950 | val 0.5062 | acc 0.725 tpr 0.800 tnr 0.650 | 3s
  ep  50 | train 0.1365 | val 0.5597 | acc 0.700 tpr 0.700 tnr 0.700 | 3s
  early stop 55, best 40
  >>> TEST Acc=67.5% TPR=65.0% TNR=70.0% | params=44,834 train=3.0min inf=1.97ms 0.079GFLOPs best_ep=40


In [36]:
results["cerebral_wm_pet"].append(
    run_seed(7, VisionMambaModel, LOADERS["cerebral_wm"][1], False, "v7_cerebralwm_pet"))


--- v7_cerebralwm_pet seed 7 ---
  ep   1 | train 0.6849 | val 0.6588 | acc 0.725 tpr 0.550 tnr 0.900 | 3s
  ep  10 | train 0.6518 | val 0.6258 | acc 0.700 tpr 0.550 tnr 0.850 | 3s
  ep  20 | train 0.6032 | val 0.5983 | acc 0.700 tpr 0.500 tnr 0.900 | 3s
  ep  30 | train 0.4603 | val 0.5529 | acc 0.750 tpr 0.500 tnr 1.000 | 3s
  ep  40 | train 0.2844 | val 0.5158 | acc 0.775 tpr 0.700 tnr 0.850 | 3s
  ep  50 | train 0.1613 | val 0.5129 | acc 0.725 tpr 0.700 tnr 0.750 | 3s
  ep  60 | train 0.1292 | val 0.5640 | acc 0.725 tpr 0.500 tnr 0.950 | 3s
  early stop 67, best 52
  >>> TEST Acc=70.0% TPR=70.0% TNR=70.0% | params=44,834 train=3.7min inf=2.10ms 0.079GFLOPs best_ep=52


In [37]:
results["cerebral_wm_pet"].append(
    run_seed(123, VisionMambaModel, LOADERS["cerebral_wm"][1], False, "v7_cerebralwm_pet"))


--- v7_cerebralwm_pet seed 123 ---
  ep   1 | train 0.6883 | val 0.6690 | acc 0.700 tpr 0.450 tnr 0.950 | 3s
  ep  10 | train 0.6392 | val 0.6227 | acc 0.700 tpr 0.450 tnr 0.950 | 3s
  ep  20 | train 0.5653 | val 0.5863 | acc 0.700 tpr 0.450 tnr 0.950 | 3s
  ep  30 | train 0.3821 | val 0.5464 | acc 0.725 tpr 0.500 tnr 0.950 | 3s
  ep  40 | train 0.1940 | val 0.5132 | acc 0.850 tpr 0.700 tnr 1.000 | 3s
  ep  50 | train 0.1357 | val 0.5135 | acc 0.850 tpr 0.700 tnr 1.000 | 3s
  ep  60 | train 0.1279 | val 0.5188 | acc 0.825 tpr 0.700 tnr 0.950 | 3s
  early stop 69, best 54
  >>> TEST Acc=62.5% TPR=60.0% TNR=65.0% | params=44,834 train=3.8min inf=1.92ms 0.079GFLOPs best_ep=54


In [39]:
results["cerebral_wm_mm"].append(
    run_seed(1, MultimodalVisionMambaModel, LOADERS["cerebral_wm"][2], True, "v7_cerebralwm_mm"))


--- v7_cerebralwm_mm seed 1 ---
  ep   1 | train 0.6955 | val 0.6692 | acc 0.675 tpr 0.750 tnr 0.600 | 6s
  ep  10 | train 0.6403 | val 0.6190 | acc 0.725 tpr 0.550 tnr 0.900 | 6s
  ep  20 | train 0.5640 | val 0.5721 | acc 0.750 tpr 0.650 tnr 0.850 | 6s
  ep  30 | train 0.3537 | val 0.5308 | acc 0.700 tpr 0.700 tnr 0.700 | 6s
  ep  40 | train 0.1727 | val 0.5291 | acc 0.750 tpr 0.650 tnr 0.850 | 6s
  ep  50 | train 0.1268 | val 0.5253 | acc 0.725 tpr 0.650 tnr 0.800 | 6s
  early stop 51, best 36
  >>> TEST Acc=75.0% TPR=60.0% TNR=90.0% | params=89,666 train=5.4min inf=3.85ms 0.158GFLOPs best_ep=36


In [40]:
results["cerebral_wm_mm"].append(
    run_seed(7, MultimodalVisionMambaModel, LOADERS["cerebral_wm"][2], True, "v7_cerebralwm_mm"))


--- v7_cerebralwm_mm seed 7 ---
  ep   1 | train 0.6922 | val 0.6672 | acc 0.700 tpr 0.700 tnr 0.700 | 6s
  ep  10 | train 0.6394 | val 0.6132 | acc 0.725 tpr 0.550 tnr 0.900 | 7s
  ep  20 | train 0.5583 | val 0.5797 | acc 0.700 tpr 0.450 tnr 0.950 | 7s
  ep  30 | train 0.3997 | val 0.5201 | acc 0.800 tpr 0.700 tnr 0.900 | 16s
  ep  40 | train 0.1710 | val 0.5290 | acc 0.725 tpr 0.700 tnr 0.750 | 16s
  early stop 46, best 31
  >>> TEST Acc=70.0% TPR=60.0% TNR=80.0% | params=89,666 train=8.2min inf=10.36ms 0.158GFLOPs best_ep=31


In [41]:
results["cerebral_wm_mm"].append(
    run_seed(123, MultimodalVisionMambaModel, LOADERS["cerebral_wm"][2], True, "v7_cerebralwm_mm"))


--- v7_cerebralwm_mm seed 123 ---
  ep   1 | train 0.6906 | val 0.6678 | acc 0.600 tpr 0.650 tnr 0.550 | 16s
  ep  10 | train 0.6410 | val 0.6194 | acc 0.725 tpr 0.550 tnr 0.900 | 16s
  ep  20 | train 0.5661 | val 0.5688 | acc 0.800 tpr 0.650 tnr 0.950 | 16s
  ep  30 | train 0.3925 | val 0.5077 | acc 0.825 tpr 0.700 tnr 0.950 | 16s
  ep  40 | train 0.1927 | val 0.5106 | acc 0.800 tpr 0.700 tnr 0.900 | 16s
  early stop 49, best 34
  >>> TEST Acc=65.0% TPR=60.0% TNR=70.0% | params=89,666 train=13.1min inf=9.89ms 0.158GFLOPs best_ep=34


In [7]:
INCLUDE_SEEDS = [1, 7, 123]

def summarize(rs, name, include=INCLUDE_SEEDS):
    rs = [r for r in rs if r['seed'] in include]
    if not rs:
        print(f"{name:28s} no runs"); return
    a  = [r['acc'] for r in rs]; t = [r['tpr'] for r in rs]; n = [r['tnr'] for r in rs]
    tm = [r['train_time_sec'] for r in rs]; inf = [r['inf_time_ms'] for r in rs]
    sd = lambda v: np.std(v, ddof=1) * 100 if len(v) > 1 else 0.0
    print(f"{name:28s} Acc={np.mean(a)*100:5.1f}±{sd(a):4.1f}% | "
          f"TPR={np.mean(t)*100:5.1f}±{sd(t):4.1f}% | "
          f"TNR={np.mean(n)*100:5.1f}±{sd(n):4.1f}% | "
          f"{rs[0]['n_params']:>7,}p | {np.mean(tm)/60:4.1f}m | "
          f"{np.mean(inf):5.2f}ms | seeds={[r['seed'] for r in rs]}")

print(f"=== v7 region-pair Vision Mamba — {TAG} augmentation, 200 subjects ===")
print( "    2 ROIs per model, 1,024 tokens (vs 3,072 for all six)")

for region in REGION_PAIRS:
    for mod in ("mri", "pet", "mm"):
        summarize(results[f"{region}_{mod}"], f"{region} / {mod}")
    print()

with open(RESULTS, 'w') as f:
    json.dump({k: [{kk: (float(vv) if isinstance(vv, (float, np.floating)) else vv)
                    for kk, vv in r.items()} for r in v] for k, v in results.items()},
              f, indent=2)
print(f"\nsaved {RESULTS}")

=== v7 region-pair Vision Mamba — tio augmentation, 200 subjects ===
    2 ROIs per model, 1,024 tokens (vs 3,072 for all six)
hippocampus / mri            Acc= 61.7± 1.4% | TPR= 66.7± 7.6% | TNR= 56.7±10.4% |  44,834p |  2.4m |  2.73ms | seeds=[1, 7, 123]
hippocampus / pet            Acc= 66.7± 2.9% | TPR= 61.7±14.4% | TNR= 71.7±12.6% |  44,834p |  2.4m |  2.62ms | seeds=[1, 7, 123]
hippocampus / mm             Acc= 72.5± 0.0% | TPR= 71.7± 7.6% | TNR= 73.3± 7.6% |  89,666p |  4.4m |  3.74ms | seeds=[1, 7, 123]

cerebellum_wm / mri          Acc= 59.2± 7.6% | TPR= 56.7± 7.6% | TNR= 61.7±10.4% |  44,834p |  3.0m |  2.08ms | seeds=[1, 7, 123]
cerebellum_wm / pet          Acc= 61.7± 2.9% | TPR= 51.7± 5.8% | TNR= 71.7± 5.8% |  44,834p |  2.4m |  1.94ms | seeds=[1, 7, 123]
cerebellum_wm / mm           Acc= 60.8± 3.8% | TPR= 60.0± 8.7% | TNR= 61.7± 5.8% |  89,666p |  4.1m |  3.78ms | seeds=[1, 7, 123]

cerebral_wm / mri            Acc= 60.8± 2.9% | TPR= 50.0±10.0% | TNR= 71.7± 5.8% |  44,834p

In [8]:
#  GFLOPs for one forward pass, batch size 1 -- region-pair models
#  (2 ROIs, 1,024 tokens)

#  Architecture alone determines these figures, so no training, checkpoints
#  or data are needed.

import copy
from torch.utils.flop_counter import FlopCounterMode
from mambapy.vim import VMamba as _VMamba


def count_flops(model, inputs):
    """Returns (conv_linear, scan, n_tokens) for one forward pass."""
    m = copy.deepcopy(model).eval().cpu()
    inputs = [t.detach().cpu() for t in inputs]
    mamba_log, handles = [], []

    def mk(mod):
        def hook(_, inp, __):
            c = mod.config
            mamba_log.append(dict(L=inp[0].shape[1], ed=c.d_inner, n=c.d_state,
                                  layers=c.n_layers,
                                  bi=getattr(c, "bidirectional", False)))
        return hook
    for mod in m.modules():                      # sequence length per encoder
        if isinstance(mod, _VMamba):
            handles.append(mod.register_forward_hook(mk(mod)))

    counter = FlopCounterMode(display=False)
    with torch.no_grad(), counter:
        m(*inputs)
    for h in handles:
        h.remove()

    cl = counter.get_total_flops()
    sc = sum(6 * r["ed"] * r["n"] * r["L"] * r["layers"] * (2 if r["bi"] else 1)
             for r in mamba_log)
    tokens = mamba_log[0]["L"] if mamba_log else 0
    del m
    return cl, sc, tokens


def report(name, model, inputs):
    p = sum(q.numel() for q in model.parameters() if q.requires_grad)
    cl, sc, tok = count_flops(model, inputs)
    print(f"  {name:34s} {p:>8,}p | {tok:,} tokens | "
          f"conv+linear {cl/1e9:.3f} + scan {sc/1e9:.3f} = {(cl+sc)/1e9:.3f} GFLOPs")


print("GFLOPs, one forward pass, batch size 1 -- region-pair models\n")
pair = lambda: torch.randn(1, 2, 1, 64, 64, 64)
report("Single ROI pair (unimodal)",   VisionMambaModel(n_rois=2).cpu(),           [pair()])
report("Single ROI pair (multimodal)", MultimodalVisionMambaModel(n_rois=2).cpu(), [pair(), pair()])


GFLOPs, one forward pass, batch size 1 -- region-pair models

  Single ROI pair (unimodal)           44,834p | 1,024 tokens | conv+linear 0.088 + scan 0.025 = 0.113 GFLOPs
  Single ROI pair (multimodal)         89,666p | 1,024 tokens | conv+linear 0.176 + scan 0.050 = 0.227 GFLOPs
